# Cliff Walking: Q-Learning vs SARSA Comparison

This notebook demonstrates the key differences between **Q-Learning** (off-policy) and **SARSA** (on-policy) reinforcement learning algorithms using the classic Cliff Walking environment.

## Learning Objectives
- Understand the Cliff Walking environment
- Implement Q-Learning and SARSA algorithms
- Compare the behavioral differences between on-policy and off-policy methods
- Visualize the learned policies

## 1. Import Required Libraries

We'll use NumPy for numerical computations and array operations.

In [1]:
import numpy as np

## 2. Environment Setup

The Cliff Walking environment is a classic reinforcement learning problem:

- **Grid**: 4×12 grid world
- **Start**: Bottom-left corner (3,0)
- **Goal**: Bottom-right corner (3,11)
- **Cliff**: Bottom row from (3,1) to (3,10)

The agent must navigate from start to goal while avoiding the cliff.

In [2]:
def cliff_walking_environment():
    """
    Simple Cliff Walking Environment
    Grid: 4x12 (4 rows, 12 columns)
    Start: (3,0) - bottom left
    Goal: (3,11) - bottom right  
    Cliff: (3,1) to (3,10) - bottom row middle
    """
    rows, cols = 4, 12
    start = (3, 0)
    goal = (3, 11)
    cliff = [(3, j) for j in range(1, 11)]  # Positions (3,1) to (3,10)

    return rows, cols, start, goal, cliff

## 3. Reward Function

The reward structure is crucial for learning:
- **Goal reached**: 0 (episode terminates successfully)
- **Fell off cliff**: -100 (large penalty + reset to start)
- **Normal step**: -1 (encourages shorter paths)

In [3]:
def get_reward(next_state: tuple, goal: tuple, cliff: list) -> int:
    """Calculate reward for transition
    
    Args:
        next_state (tuple): The state resulting from the action
        goal (tuple): The goal state
        cliff (list): List of cliff states

    Returns:
        An integer reward based on the next state:
        - 0 if next_state is the goal (successful episode)
        - -100 if next_state is in the cliff (fell off)
        - -1 for any other state (normal step cost)
    """
    reward = -1
    if next_state == goal:
        reward = 0
    elif next_state in cliff:
        reward = -100
    return reward

## 4. State Transition Function

This function handles agent movement in the grid world:
- **Actions**: 0=RIGHT, 1=LEFT, 2=DOWN, 3=UP
- **Boundaries**: Agent cannot move outside the grid
- **Deterministic**: Each action always produces the same result

In [4]:
def get_next_state(state: tuple, action: int, rows: int, cols: int) -> tuple:
    """Execute action and return next state
    
    Args:
        state (tuple): Current state (row, col)
        action (int): Action index (0=RIGHT, 1=LEFT, 2=DOWN, 3=UP)
        rows (int): Number of rows in the grid
        cols (int): Number of columns in the grid

    Returns:
        A tuple representing the new state after taking the action,
        ensuring that the agent does not move outside the grid boundaries.
    """
    actions = [(0,1), (0, -1), (1, 0), (-1, 0)]
    row, col = state
    dr, dc = actions[action]

    new_row = max(0, min(rows - 1, row + dr))
    new_col = max(0, min(cols - 1, col + dc))

    return new_row, new_col

## 5. Action Selection Strategy

**Epsilon-Greedy Policy**: Balances exploration vs exploitation
- **Exploration** (ε probability): Choose random action
- **Exploitation** (1-ε probability): Choose best known action

This strategy is used by both Q-Learning and SARSA algorithms.

In [5]:
def epsilon_greedy(Q: np.ndarray, state: tuple, epsilon: float, num_actions: int) -> int:
    """Epsilon-greedy action selection
    
    Args:
        Q (np.ndarray): Q-table with shape (rows, cols, num_actions)
        state (tuple): Current state (row, col)
        epsilon (float): Exploration rate (0 to 1)
        num_actions (int): Total number of possible actions

    Returns:
        An integer representing the chosen action index:
        - With probability epsilon, a random action is chosen (exploration).
        - With probability (1 - epsilon), the action with the highest Q-value for the current
    """
    if np.random.random() < epsilon:
        return np.random.choice(num_actions) #exploration
    else:
        return np.argmax(Q[state]) # exploitation

## 6. Q-Learning Algorithm (Off-Policy)

**Q-Learning** is an off-policy algorithm:
- Updates Q-values using the **maximum** Q-value of the next state
- Learns the **optimal policy** regardless of the behavior policy
- **Bold**: Tends to learn aggressive policies (closer to cliff)

**Update Rule**: Q(s,a) ← Q(s,a) + α[r + γ max Q(s',a') - Q(s,a)]

In [6]:
def q_learning_cliff(episodes=500, alpha=0.1, gamma=1.0, epsilon=0.1):
    """Q-Learning for Cliff Walking"""
    rows, cols, start, goal, cliff = cliff_walking_environment()
    Q = np.zeros((rows, cols, 4)) # states vs. actions
    
    for episode in range(episodes):
        state = start
        
        while state != goal:
            # Choose action using epsilon-greedy
            action = epsilon_greedy(Q, state, epsilon, 4)
            
            # Take action
            next_state = get_next_state(start, action, rows, cols)
            reward = get_reward(next_state, goal, cliff)
            
            # Q-Learning update (OFF-POLICY)
            best_next_action = np.argmax( Q[next_state] ) # action with the highest reward in next_state
            Q[state][action] +=  alpha * (reward + (gamma * Q[next_state][best_next_action]) - Q[state][action])
            
            # Reset to start if fell off cliff
            if next_state in cliff:
                state = start
            else:
                state = next_state
    
    return Q

## 7. SARSA Algorithm (On-Policy)

**SARSA** (State-Action-Reward-State-Action) is an on-policy algorithm:
- Updates Q-values using the **actual next action** to be taken
- Learns the policy it's **currently following**
- **Cautious**: Tends to learn safer policies (away from cliff)

**Update Rule**: Q(s,a) ← Q(s,a) + α[r + γ Q(s',a') - Q(s,a)]

In [7]:
def sarsa_cliff(episodes=500, alpha=0.1, gamma=1.0, epsilon=0.1):
    """
    """
    rows, cols, start, goal, cliff = cliff_walking_environment()
    Q = np.zeros((rows, cols, 4))

    for episode in range(episodes):
        state = start
        action = epsilon_greedy(Q, state, epsilon, 4)

        while state != goal:
            next_state = get_next_state(state, action, rows, cols)
            reward = get_reward(state, goal, cliff)

            next_action = epsilon_greedy(Q, next_state, epsilon, 4)
            Q[state][action] +=  alpha * (reward + (gamma * Q[next_state][next_action]) - Q[state][action])

            # Reset to start if fell off cliff
            if next_state in cliff:
                state = start
                action = epsilon_greedy(Q, state, epsilon, 4)
            else:
                state = next_state
                action = next_action
    
    return Q
            

---
## Utilities

In [8]:
def extract_path(Q, start, goal, cliff, rows, cols):
    """Extract path from learned policy"""
    state = start
    path = [state]
    
    print(f"Tracing path from {start} to {goal}:")
    
    for step in range(50):  # Prevent infinite loops, increased limit
        if state == goal:
            print(f"Step {step}: Reached goal at {state}")
            break
        action = np.argmax(Q[state])
        next_state = get_next_state(state, action, rows, cols)
        path.append(next_state)
        
        action_names = ["RIGHT", "LEFT", "DOWN", "UP"]
        print(f"Step {step+1}: From {state} take {action_names[action]} to {next_state}")
        
        if next_state in cliff:
            print(f"Hit cliff at {next_state}!")
            break
        state = next_state
    
    print(f"Complete path: {path}")
    return path

In [9]:
def print_path(path, rows, cols, cliff, start, goal):
    """Print the path on a grid"""
    grid = [['.' for _ in range(cols)] for _ in range(rows)]
    
    # Mark cliff
    for cr, cc in cliff:
        grid[cr][cc] = 'C'
    
    # Mark start and goal
    grid[start[0]][start[1]] = 'S'
    grid[goal[0]][goal[1]] = 'G'
    
    # Mark path
    for i, (pr, pc) in enumerate(path[1:-1], 1):
        if grid[pr][pc] == '.':
            grid[pr][pc] = str(i % 10)
    
    # Print grid
    for row in grid:
        print(' '.join(row))
    print()

## 9. Main Comparison and Results

Now let's run both algorithms and compare their learned behaviors!

In [10]:
if __name__ == "__main__":
    print("CLIFF WALKING COMPARISON")
    print("=" * 40)
    print("Grid: 4x12, S=Start, G=Goal, C=Cliff")
    print("Path shows learning result\n")
    
    # Run Q-Learning
    print("Q-LEARNING RESULT:")
    Q_q = q_learning_cliff()
    rows, cols, start, goal, cliff = cliff_walking_environment()
    path_q = extract_path(Q_q, start, goal, cliff, rows, cols)
    print_path(path_q, rows, cols, cliff, start, goal)
    print(f"Q-Learning path length: {len(path_q)-1} steps\n")
    
    # Run SARSA
    print("SARSA RESULT:")
    Q_s = sarsa_cliff()
    path_s = extract_path(Q_s, start, goal, cliff, rows, cols)
    print_path(path_s, rows, cols, cliff, start, goal)
    print(f"SARSA path length: {len(path_s)-1} steps")
    
    print("\nKEY DIFFERENCE:")
    print("- Q-Learning: Takes risky optimal path (close to cliff)")
    print("- SARSA: Takes safer path (away from cliff)")

CLIFF WALKING COMPARISON
Grid: 4x12, S=Start, G=Goal, C=Cliff
Path shows learning result

Q-LEARNING RESULT:


KeyboardInterrupt: 

## 10. Key Learning Points

### Algorithm Differences

| Aspect | Q-Learning (Off-Policy) | SARSA (On-Policy) |
|--------|------------------------|-------------------|
| **Update Rule** | Uses max Q(s',a') | Uses actual Q(s',a') |
| **Policy** | Learns optimal policy | Learns current policy |
| **Risk Behavior** | More aggressive/risky | More conservative/safe |
| **Cliff Walking** | Path close to cliff | Path away from cliff |

### Why the Difference?

- **Q-Learning**: Learns the optimal policy assuming perfect action selection, leading to risky but optimal paths
- **SARSA**: Learns while considering exploration mistakes, leading to safer paths that account for potential errors

### Practical Implications

- Use **Q-Learning** when you want the theoretically optimal policy
- Use **SARSA** when safety during learning is important
- The choice depends on whether you can afford exploration mistakes in the real environment